# Phase 2 — Baseline (Arm A)

Thin Colab runner for the **baseline** arm: the base `Qwen/Qwen3.5-4B` model with **no** retrieved context.

This notebook only clones the repo, installs pinned dependencies, and calls `scripts/generate.py` + `scripts/score.py`. All logic lives in the repo.

**Runtime:** set it to a GPU — target **L4 (24 GB)**, fallback **A100**. The model is loaded in bf16 (no 4-bit) with thinking disabled.

In [ ]:
# Confirm the GPU (expect an L4 or A100).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo (idempotent) and enter it.
import os
REPO = 'https://github.com/yusuftanriverdi24/Polars-Rag---Finetune.git'
if not os.path.isdir('polars-rag-vs-ft'):
    !git clone -q $REPO polars-rag-vs-ft
%cd polars-rag-vs-ft
!git pull -q

In [ ]:
# Install pinned dependencies: the `colab` extra brings the pinned GPU stack
# (unsloth/transformers/peft/trl/accelerate); `-e .` pins Polars to 2.0.0 and
# installs the package. torch is left as Colab's preinstalled CUDA build.
!pip install -q -e ".[colab]"

In [ ]:
# Generate baseline outputs for all 120 benchmark tasks (GPU).
!python scripts/generate.py --arm A --model Qwen/Qwen3.5-4B --batch-size 8

In [ ]:
# Score the generations on CPU (sandbox execution under Polars 2.0.0).
!python scripts/score.py --arm A

In [ ]:
# Summary: metrics block from the scores file.
!python -m json.tool results/A/scores.json | sed -n '1,60p'

In [ ]:
# Zip results for download.
!zip -q -r results_A.zip results/A
from google.colab import files
files.download('results_A.zip')